# SVD Collaborative Filtering Baseline

This notebook establishes a collaborative-filtering baseline for the Bayesian meal recommendation project.

The model uses **SVD matrix factorization** to predict explicit 1–5 recipe ratings from user–recipe interaction history. It loads the model-ready ratings produced by the data cleaning and EDA notebook, creates a reproducible temporal holdout, fits the SVD model, evaluates held-out rating predictions, and saves the split and results for later comparison with the Bayesian recommender.

### Goals
- Use the existing processed explicit ratings without repeating data cleaning or EDA.
- Evaluate future-like predictions using each eligible user's most recent rating as the test observation.
- Report RMSE and MAE for the SVD baseline.
- Track coverage for recipes seen during training.
- Save the exact train/test split so later models can be evaluated on the same observations.


## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from surprise import Dataset, Reader, SVD
except ImportError as exc:
    raise ImportError(
        "scikit-surprise is required for this notebook. "
        "Install it with: pip install scikit-surprise"
    ) from exc

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "reports" / "model_results"

RATINGS_PATH = PROCESSED_DIR / "explicit_ratings.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Ratings path:", RATINGS_PATH)


## 2. Load Model-Ready Ratings

The cleaning/EDA notebook already removed zero-coded interactions from the explicit-rating dataset. This notebook therefore loads `explicit_ratings.csv` directly and uses only the columns needed for collaborative filtering.


In [ ]:
ratings = pd.read_csv(
    RATINGS_PATH,
    usecols=["user_id", "recipe_id", "date", "rating"],
    parse_dates=["date"],
)

print(f"Ratings: {len(ratings):,}")
print(f"Users: {ratings['user_id'].nunique():,}")
print(f"Recipes: {ratings['recipe_id'].nunique():,}")
print(f"Rating range: {ratings['rating'].min()}–{ratings['rating'].max()}")

ratings.head()


In [ ]:
validation = pd.Series({
    "rows": len(ratings),
    "missing_user_id": ratings["user_id"].isna().sum(),
    "missing_recipe_id": ratings["recipe_id"].isna().sum(),
    "missing_rating": ratings["rating"].isna().sum(),
    "missing_date": ratings["date"].isna().sum(),
    "duplicate_rows": ratings.duplicated().sum(),
    "ratings_outside_1_to_5": (~ratings["rating"].between(1, 5)).sum(),
})

validation


## 3. Temporal Train/Test Split

A random interaction split can allow later behavior to help predict earlier behavior. Instead, this baseline uses a **per-user temporal holdout**:

- Users with only one rating remain entirely in training because no earlier history exists for them.
- For each user with at least two ratings, the most recent interaction is held out for testing.
- All earlier interactions are used for training.

This design evaluates whether the model can predict a user's later preference from their prior ratings while retaining the sparse users as useful recipe-level evidence in training.


In [ ]:
ratings = ratings.sort_values(
    ["user_id", "date", "recipe_id"]
).reset_index(drop=True)

user_rating_count = ratings.groupby("user_id")["rating"].transform("size")
eligible_mask = user_rating_count >= 2

test_idx = (
    ratings.loc[eligible_mask]
    .groupby("user_id", sort=False)
    .tail(1)
    .index
)

test_df = ratings.loc[test_idx].copy()
train_df = ratings.drop(index=test_idx).copy()

print(f"Training ratings: {len(train_df):,}")
print(f"Test ratings: {len(test_df):,}")
print(f"Users represented in test: {test_df['user_id'].nunique():,}")
print(f"Train + test = original: {len(train_df) + len(test_df) == len(ratings)}")


In [ ]:
train_users = set(train_df["user_id"])
train_recipes = set(train_df["recipe_id"])

test_df["known_user"] = test_df["user_id"].isin(train_users)
test_df["known_recipe"] = test_df["recipe_id"].isin(train_recipes)
test_df["known_user_and_recipe"] = test_df["known_user"] & test_df["known_recipe"]

coverage_summary = pd.Series({
    "test_rows": len(test_df),
    "known_user_pct": test_df["known_user"].mean() * 100,
    "known_recipe_pct": test_df["known_recipe"].mean() * 100,
    "known_user_and_recipe_pct": test_df["known_user_and_recipe"].mean() * 100,
}).round(2)

coverage_summary


The primary SVD comparison should focus on test observations for which both the user and recipe are represented in training. Predictions for unseen recipes are still reported separately because they illustrate the collaborative-filtering cold-start limitation.


## 4. Simple Mean Benchmark

Before fitting SVD, a global-mean predictor provides a minimal sanity-check benchmark. SVD should improve on this naive prediction if collaborative structure is informative.


In [ ]:
global_mean = train_df["rating"].mean()
mean_predictions = np.repeat(global_mean, len(test_df))

mean_rmse = np.sqrt(mean_squared_error(test_df["rating"], mean_predictions))
mean_mae = mean_absolute_error(test_df["rating"], mean_predictions)

print(f"Training global mean: {global_mean:.4f}")
print(f"Global mean RMSE: {mean_rmse:.4f}")
print(f"Global mean MAE: {mean_mae:.4f}")


## 5. Fit SVD Collaborative Filtering Model

The SVD recommender estimates a rating as

\[
\hat r_{ui} = \mu + b_u + b_i + q_i^\top p_u,
\]

where \(\mu\) is the global mean, \(b_u\) and \(b_i\) are user and recipe biases, and \(p_u\) and \(q_i\) are learned latent-factor vectors.

The hyperparameters below are intentionally fixed for the baseline. Extensive tuning can be added later if needed, but the initial purpose is to establish a clear benchmark for the Bayesian model.


In [ ]:
reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(
    train_df[["user_id", "recipe_id", "rating"]],
    reader,
)
trainset = train_data.build_full_trainset()

svd = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=RANDOM_STATE,
)

svd.fit(trainset)


## 6. Predict Held-Out Ratings

In [ ]:
svd_predictions = [
    svd.predict(row.user_id, row.recipe_id)
    for row in test_df.itertuples(index=False)
]

test_results = test_df.copy()
test_results["svd_prediction"] = [pred.est for pred in svd_predictions]
test_results["prediction_error"] = (
    test_results["rating"] - test_results["svd_prediction"]
)

test_results.head()


## 7. Evaluate Baseline Performance

In [ ]:
def regression_metrics(df, prediction_col):
    rmse = np.sqrt(mean_squared_error(df["rating"], df[prediction_col]))
    mae = mean_absolute_error(df["rating"], df[prediction_col])
    return rmse, mae


svd_rmse_all, svd_mae_all = regression_metrics(
    test_results,
    "svd_prediction",
)

known_test = test_results.loc[
    test_results["known_user_and_recipe"]
].copy()

svd_rmse_known, svd_mae_known = regression_metrics(
    known_test,
    "svd_prediction",
)

metrics = pd.DataFrame({
    "model": [
        "Global mean",
        "SVD - all held-out ratings",
        "SVD - known user & recipe",
    ],
    "n_test": [
        len(test_results),
        len(test_results),
        len(known_test),
    ],
    "rmse": [
        mean_rmse,
        svd_rmse_all,
        svd_rmse_known,
    ],
    "mae": [
        mean_mae,
        svd_mae_all,
        svd_mae_known,
    ],
}).round(4)

metrics


In [ ]:
error_summary = test_results["prediction_error"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).round(4)

error_summary


### Interpretation

Use the executed results to summarize:

- Whether SVD improves on the global-mean benchmark.
- The RMSE and MAE on the full temporal holdout.
- The RMSE and MAE when both the user and recipe were observed during training.
- The percentage of test recipes that are unseen during training.

The known-user-and-recipe result is the cleanest measure of SVD's collaborative-filtering performance. The gap between that result and the full test result helps quantify the cold-start limitation that the later Bayesian/hybrid model should address.


## 8. Save Split and Baseline Results

The exact split is saved so the Bayesian model can be evaluated against the same held-out interactions. This prevents differences in train/test composition from being mistaken for differences in model quality.


In [ ]:
split_columns = ["user_id", "recipe_id", "date", "rating"]

train_df[split_columns].to_csv(
    PROCESSED_DIR / "svd_train.csv",
    index=False,
)

test_df[split_columns].to_csv(
    PROCESSED_DIR / "svd_test.csv",
    index=False,
)

test_results.to_csv(
    RESULTS_DIR / "svd_test_predictions.csv",
    index=False,
)

metrics.to_csv(
    RESULTS_DIR / "svd_baseline_metrics.csv",
    index=False,
)

print("Saved:")
print(PROCESSED_DIR / "svd_train.csv")
print(PROCESSED_DIR / "svd_test.csv")
print(RESULTS_DIR / "svd_test_predictions.csv")
print(RESULTS_DIR / "svd_baseline_metrics.csv")


## 9. Baseline Summary

SVD serves as the project's conventional collaborative-filtering benchmark. It learns latent user and recipe representations from explicit rating history but does not directly use the engineered recipe attributes or quantify posterior uncertainty. Those limitations provide a clear comparison point for the Bayesian recommendation model.

The next modeling stage should reuse `svd_train.csv` and `svd_test.csv` so that both approaches are evaluated on the same temporal holdout.
